# 품질 표시와 판단할 값

실제 과제 코드의 실행·DB 재조회 결과입니다. UCI 원본의 오류 주입 복사본과 교육용 시뮬레이터를 구분합니다.

In [1]:
import sys, json
from pathlib import Path
import pandas as pd
import numpy as np
sys.path.insert(0, '/course/플랫폼코드')
import quality_lab as lab
import quality as practical
import practical_tests
from hydops.b1_data import uci
from hydops.b2_quality.checks import window_quality, classify_window

original=uci.cycle_observations(uci.load_reduced(),1500,'HYD-01',lab.REPLAY_START)
injected=lab.make_injected_rows()
checked=lab.check_rows(injected)
other=practical.LabQualityChecker().check_many(injected)
assert checked==other, '반별 실제 품질 결과 불일치'
run_id=lab.new_run_id()
inserted=lab.load_to_db(run_id,checked)
assert inserted==180
fetched=lab.last_60s(run_id)
assert len(fetched)==60 and {r['origin_cycle_id'] for r in fetched}=={1500}
assert sum(r['raw_value'] is None for r in fetched)==7
assert sum(r['value'] is None for r in fetched)==11
print('DB 실행:',run_id,'/ HYD-01 / 원본 사이클1500 / 적재180행 / TS1 조회60행')


DB 실행: LAB-S02-6aee5060 / HYD-01 / 원본 사이클1500 / 적재180행 / TS1 조회60행


In [2]:
# 원본과 오류 주입 복사본
o={r['elapsed_s']:r['raw_value'] for r in original if r['sensor_id']=='TS1'}
i={r['elapsed_s']:r['raw_value'] for r in injected if r['sensor_id']=='TS1'}
before=[{'경과 초':s,'원본 온도':o[s],'오류 주입 후':i[s]} for s in [18,19,20,25,26,27,42,43,44]]
print('TS1 / °C / 사이클1500 · 원본 파일은 바꾸지 않았습니다.')
display(pd.DataFrame(before).fillna('비어 있음'))


TS1 / °C / 사이클1500 · 원본 파일은 바꾸지 않았습니다.


,경과 초,원본 온도,오류 주입 후
0,18,35.410,35.41
1,19,35.414,60.414
2,20,35.422,35.422
3,25,35.426,35.426
4,26,35.500,비어 있음
5,27,35.434,비어 있음
6,42,35.668,35.668
7,43,35.652,35.652
8,44,35.574,35.652


In [3]:
# 품질 표시와 DB에 남은 값
selected=[r for r in fetched if r['elapsed_s'] in [18,19,20,26,29,30,32,49,50,52,53]]
display(pd.DataFrame([{'경과 초':r['elapsed_s'],'검사 전 값':r['raw_value'],
                       '판단할 값':r['value'],'품질 표시':r['quality_flag']} for r in selected]).fillna('비어 있음'))
counts=[{'센서':sid,**lab.flag_counts(checked,sid)} for sid in ['TS1','PS1','FS1']]
print('전체60초의 센서별 표시 개수')
display(pd.DataFrame(counts).fillna(0).set_index('센서').astype(int))


,경과 초,검사 전 값,판단할 값,품질 표시
0,18,35.41,35.41,OK
1,19,60.414,비어 있음,SPIKE
2,20,35.422,35.422,OK
3,26,비어 있음,비어 있음,MISSING
4,29,비어 있음,비어 있음,MISSING
5,30,비어 있음,비어 있음,GAP
6,32,비어 있음,비어 있음,GAP
7,49,35.652,35.652,OK
8,50,35.652,비어 있음,STUCK
9,52,35.652,비어 있음,STUCK


전체60초의 센서별 표시 개수


,OK,SPIKE,MISSING,GAP,STUCK
센서,,,,,
TS1,49,1,4,3,3
PS1,60,0,0,0,0
FS1,60,0,0,0,0


In [4]:
# 같은 실행에서 구간별 판단
ranges=[('전체',0,59),('비교 A',5,9),('비교 B',17,21),('비교 C',28,32)]
range_results=[]
for label,start,end in ranges:
    part=lab.range_rows(run_id,(start,end))
    q=window_quality(part)
    range_results.append({'구간':label,'경과 초':f'{start}~{end}','관측 수':q['n'],
                          '품질 표시':str(q['flags']),'결과':classify_window(part,lab.RULES)})
assert [r['결과'] for r in range_results]==['SENSOR_FAULT','VALID','VALID','SENSOR_FAULT']
display(pd.DataFrame(range_results))
print('VALID는 선택한 구간의 센서 품질 결과입니다. 설비 정상 판정이 아닙니다.')


,구간,경과 초,관측 수,품질 표시,결과
0,전체,0~59,60,"{'OK': 49, 'SPIKE': 1, 'MISSING': 4, 'GAP': 3,...",SENSOR_FAULT
1,비교 A,5~9,5,{'OK': 5},VALID
2,비교 B,17~21,5,"{'OK': 4, 'SPIKE': 1}",VALID
3,비교 C,28~32,5,"{'MISSING': 2, 'GAP': 3}",SENSOR_FAULT


VALID는 선택한 구간의 센서 품질 결과입니다. 설비 정상 판정이 아닙니다.


In [5]:
# 결측 샘플의 1초 집계
from hydops.config import DATA_DIR
from itertools import islice
with open(DATA_DIR/'raw/PS1.txt') as f:
    samples=np.array(next(islice(f,100,101)).split(),dtype=float)
modified=samples.copy()
modified[500:600]=np.nan
modified[600:670]=np.nan
modified[700:720]=np.nan
seconds=practical.aggregate_1s(modified,100)
assert [seconds[s]['n'] for s in [5,6,7,8]]==[0,30,80,100]
assert seconds[5]['mean'] is None and seconds[6]['mean'] is None
display(pd.DataFrame([{'경과 초':s,'유효 샘플':seconds[s]['n'],'평균 [bar]':seconds[s]['mean']}
                     for s in [5,6,7,8]]).fillna('비어 있음'))
print('PS1 · 원본 사이클100의 복사본에 결측 주입 · 1초당100샘플 · 최소50개 필요')


,경과 초,유효 샘플,평균 [bar]
0,5,0,비어 있음
1,6,30,비어 있음
2,7,80,190.091
3,8,100,190.1141


PS1 · 원본 사이클100의 복사본에 결측 주입 · 1초당100샘플 · 최소50개 필요


In [6]:
# 냉각 변화와 센서 고착
sim_results=[]
for label,fault in [('냉각 심각 저하',None),('냉각 저하 뒤 고착','stuck')]:
    sim_rows=practical.LabQualityChecker().check_many(practical_tests._severe_degradation(fault))
    part=sim_rows[-60:]
    state=practical.window_state(part)
    sim_results.append({'사례':label,'최근 관측 수':len(part),'SPIKE 수':sum(r['quality_flag']=='SPIKE' for r in sim_rows),
                        '최근60초 STUCK 수':sum(r['quality_flag']=='STUCK' for r in part),'결과':state['state']})
assert [r['결과'] for r in sim_results]==['EQUIPMENT_ANOMALY','SENSOR_FAULT']
display(pd.DataFrame(sim_results))
print('교육용 시뮬레이터 · 동일 시드7 · 냉각 효율0.25 · 실제 설비 측정 결과 아님')
result={'run_id':run_id,'inserted':inserted,'database_rows':fetched,'flags':counts,
        'ranges':range_results,'simulator':sim_results,'cohort_quality_outputs_equal':True}
_ = Path('검증결과.json').write_text(json.dumps(result,ensure_ascii=False,indent=2,default=str),encoding='utf-8')


,사례,최근 관측 수,SPIKE 수,최근60초 STUCK 수,결과
0,냉각 심각 저하,60,0,0,EQUIPMENT_ANOMALY
1,냉각 저하 뒤 고착,60,0,8,SENSOR_FAULT


교육용 시뮬레이터 · 동일 시드7 · 냉각 효율0.25 · 실제 설비 측정 결과 아님
